# Desarrollo V4d — qwen3.8:27b-q4_K_M

**Mismo prompt V4d y casos de Gemma 12B.** Este cuaderno guarda resultados aparte. Ejecutar primero el piloto (8 casos) y revisar las salidas antes de los 4 restantes. Los casos se reutilizan para comparar configuraciones; no constituyen evaluación final.


## 1. Verificar archivos compartidos y modelo (sin inferencias)

In [1]:
import json, hashlib, copy, sys, time, re, random
from pathlib import Path
from urllib.request import Request, build_opener, ProxyHandler
from uuid import uuid4
from datetime import datetime, timezone

CONFIG_NUEVA = "configuracion_calibracion_v4d_v1.json"
SHA_CONFIG_NUEVA = "e13dc4ff8cb3505973eb67bc50443ecff50231af3aee1142b300cf59209d2b53"
RUTAS = [Path(".."), Path("."), Path("02_Ronda2_V4d"), Path("05_Pruebas/02_Ronda2_V4d")]
BASE = next((p.resolve() for p in RUTAS if (p / CONFIG_NUEVA).is_file()), None)
assert BASE is not None, "Abre este notebook desde la carpeta del modelo o la raiz del repositorio."
b = (BASE / CONFIG_NUEVA).read_bytes()
assert hashlib.sha256(b).hexdigest() == SHA_CONFIG_NUEVA, "Cambio en configuracion V4d: NO continuar."
# V4d original identifica Gemma12B por ser el modelo de calibración.
# Aquí el perfil se obtiene de la configuración inicial de R2, sin editar V4d.
config_v4d = json.loads(b)

ORIGEN = BASE.parent / config_v4d["ruta_relativa_original"]
bo = (ORIGEN / "configuracion_ronda2_desarrollo_v1.json").read_bytes()
bc = (ORIGEN / "casos_desarrollo_ronda2_v1.json").read_bytes()
assert hashlib.sha256(bo).hexdigest() == config_v4d["original_config_sha256"], "Cambio en config V1 original."
assert hashlib.sha256(bc).hexdigest() == config_v4d["original_cases_sha256"], "Cambio en casos originales."
config_original = json.loads(bo)
casos = {c["case_id"]: c for c in json.loads(bc)["casos"]}
assert all(c["split"] == "desarrollo" and c["tipo"] == "sintetico" for c in casos.values())
PERFIL_ID = 'qwen38_27b'
PERFIL = next(p for p in config_original["modelos"] if p["perfil_id"] == PERFIL_ID)
MODELO = PERFIL["modelo"]
assert PERFIL["modelo_digest_esperado"] == '25b843619e944cd0ae6069f94ff4e5e26a16e109ccbc0a66a0f05979ed70098e', "Digest inesperado en configuración original."
SOLICITUD_BASE = copy.deepcopy(PERFIL["solicitud_base"])
assert SOLICITUD_BASE["think"] is False, "Modo thinking cambiado."
API = "http://127.0.0.1:11434"
assert config_v4d["api_url"] == API + "/api/chat"
cliente = build_opener(ProxyHandler({}))

def local_api(endpoint, datos=None, timeout=15):
    body = None if datos is None else json.dumps(datos, ensure_ascii=False).encode("utf-8")
    req = Request(API + endpoint, data=body, headers={"Content-Type":"application/json"}, method="POST" if datos is not None else "GET")
    with cliente.open(req, timeout=timeout) as f:
        return json.load(f)

version = local_api("/api/version")["version"]
assert version == config_v4d["ollama_version_esperada"], "Ollama cambió: versiona antes de continuar."
tags = {m["name"]:m for m in local_api("/api/tags")["models"]}
assert MODELO in tags and tags[MODELO]["digest"] == PERFIL["modelo_digest_esperado"], "Modelo o digest instalado diferente."

SALIDA = BASE / 'Qwen3.8-27B' / "resultados_v4d.jsonl"
SALIDA.parent.mkdir(exist_ok=True)
print("Protocolo:", config_v4d["protocolo_id"], "| Perfil de comparación:", PERFIL_ID)
print("Modelo verificado:", MODELO)
print("Ollama:", version, "| Python:",sys.version.split()[0])
print("Datos originales y configuración: SHA256 verificados")
print("Prompt:",config_v4d["prompt_version"],"| seed:",config_v4d["seed"])
print("Parámetros heredados:",SOLICITUD_BASE["options"])
print("Piloto:",config_v4d["piloto_case_ids"])
print("Registro independiente:",SALIDA)
print("Esta celda NO ejecuta inferencias.")

Protocolo: TESIS_ISBD_R2_CALIBRACION_V4D_V1 | Perfil de comparación: qwen38_27b
Modelo verificado: qwen3.8:27b-q4_K_M
Ollama: 0.40.13 | Python: 3.11.16
Datos originales y configuración: SHA256 verificados
Prompt: V4d | seed: 1729
Parámetros heredados: {'num_ctx': 4096, 'num_predict': 2048, 'temperature': 0.7, 'top_p': 0.8, 'top_k': 20, 'min_p': 0, 'presence_penalty': 1.5, 'repeat_penalty': 1}
Piloto: ['D2-001', 'D2-002', 'D2-003', 'D2-004', 'D2-009', 'D2-010', 'D2-011', 'D2-012']
Registro independiente: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\05_Pruebas\02_Ronda2_V4d\Qwen3.8-27B\resultados_v4d.jsonl
Esta celda NO ejecuta inferencias.


## 2. Revisar instrucciones y casos del piloto (sin inferencias)

In [2]:
print("V4d congelado. Mismos datos y sistema que en Gemma12B; cambia únicamente el perfil del modelo.")
print("No se envían las guías evaluacion_no_enviar.")
print("\nPROMPT V4d:\n" + config_v4d["system_prompt"])
print("\nCASOS DEL PILOTO:")
for cid in config_v4d["piloto_case_ids"]:
    c = casos[cid]
    print(cid, "|", c["grupo_evidencia"], "|", c["prediccion_recibida"])


V4d congelado. Mismos datos y sistema que en Gemma12B; cambia únicamente el perfil del modelo.
No se envían las guías evaluacion_no_enviar.

PROMPT V4d:
Evalúa exclusivamente lo que permite afirmar el paquete de evidencia sobre un par de autores y una predicción de coautoría ya recibida. Conserva la etiqueta exacta y su carácter sintético. No necesitas defender la etiqueta ni convertirla en un hecho.

Separa tres asuntos que no son equivalentes: (1) hechos de las publicaciones y del grafo; (2) pertinencia académica documentada del par; (3) causas técnicas de la predicción. El punto (2) NO debe cambiar al modificar solo la etiqueta del mismo paquete.

Valora la pertinencia académica del PAR, no la certeza de coautoría ni la corrección de la etiqueta. Utiliza exactamente una categoría:
- INSUFICIENTE: falta información sobre uno de los autores o no se observa en las fuentes un punto de contacto temático, metodológico o de colaboración documentado para el par.
- PARCIAL: hay afinidad temá

## 3. Ejecutar primero el piloto

Cambia `EJECUTAR=False` a `True` cuando termine la verificación. No actives `restantes` sin haber evaluado las ocho respuestas.


In [4]:
import json, hashlib, random, copy, time, re
from datetime import datetime, timezone
from uuid import uuid4
from urllib.request import Request

# Cambia SOLO EJECUTAR a True una vez que hayas revisado las dos celdas anteriores.
ETAPA = "restantes"
EJECUTAR = True

seleccion = (config_v4d["piloto_case_ids"] if ETAPA == "piloto" else config_v4d["restantes_case_ids"])
assert ETAPA in ("piloto", "restantes")
solicitudes = []
for cid in seleccion:
    caso = casos[cid]
    solicitud = copy.deepcopy(SOLICITUD_BASE)
    solicitud["options"]["seed"] = config_v4d["seed"]
    solicitud["messages"] = [
        {"role":"system", "content":config_v4d["system_prompt"]},
        {"role":"user", "content":config_v4d["user_prompt_template"].format(case_id=cid, contenido=caso["contenido"])}
    ]
    # El contenido evaluacion_no_enviar NO se incorpora a messages.
    assert all("evaluacion_no_enviar" not in m["content"] for m in solicitud["messages"])
    clave_obj = {"protocol":config_v4d["protocolo_id"], "case":cid, "model_digest":PERFIL["modelo_digest_esperado"], "solicitud":solicitud}
    clave = hashlib.sha256(json.dumps(clave_obj, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()
    solicitudes.append((cid,caso,solicitud,clave))

# Orden fijado para poder repetir el procedimiento, no para buscar mejores respuestas.
random.Random(config_v4d["seed"]).shuffle(solicitudes)

registros=[]
if SALIDA.exists():
    registros = [json.loads(s) for s in SALIDA.read_text(encoding="utf-8").splitlines() if s.strip()]
claves_ok = {r["llave_ensayo"] for r in registros if r.get("estado_ejecucion")=="completa"}
claves_todas = {r["llave_ensayo"] for r in registros}
pendientes = [x for x in solicitudes if x[3] not in claves_ok]
print("Etapa:",ETAPA,"| programadas:",len(solicitudes),"| ya completas:",len(solicitudes)-len(pendientes),"| pendientes:",len(pendientes))

# No repetir un intento fallido de manera implícita.
if any(x[3] in claves_todas for x in pendientes):
    raise RuntimeError("Hay una ejecución no completa: revisa el registro antes de hacer otro intento.")

if not EJECUTAR:
    print("Sin inferencias. Activa EJECUTAR=True cuando decidas iniciar este piloto.")
else:
    for cid,caso,solicitud,clave in pendientes:
        registro = {
            "execution_id":str(uuid4()), "llave_ensayo":clave,
            "protocolo_id":config_v4d["protocolo_id"], "case_id":cid,
            "grupo_evidencia":caso["grupo_evidencia"],
            "prediccion_recibida":caso["prediccion_recibida"],
            "prompt_version":"V4d", "grupo":"desarrollo_calibracion",
            "modelo":MODELO, "modelo_digest":PERFIL["modelo_digest_esperado"],
            "seed":config_v4d["seed"], "ollama_version":version,
            "sha256_config_v4d":SHA_CONFIG_NUEVA,
            "sha256_config_original":config_v4d["original_config_sha256"],
            "sha256_casos":config_v4d["original_cases_sha256"],
            "fecha_inicio_utc":datetime.now(timezone.utc).isoformat(),
            "solicitud":solicitud,
            "evaluacion_calidad":"pendiente",
            "reintentos":0
        }
        print("Ejecutando",cid,"| V4d",flush=True)
        t0=time.perf_counter()
        try:
            req=Request(config_v4d["api_url"], data=json.dumps(solicitud,ensure_ascii=False).encode('utf-8'), headers={"Content-Type":"application/json"},method="POST")
            with cliente.open(req,timeout=900) as f:
                respuesta=json.load(f)
            registro["respuesta_original"] = respuesta
            msg=respuesta.get("message",{})
            contenido=msg.get("content", "")
            original_tiene_think=bool(re.search(r"<\s*/?\s*think\b",contenido,re.I))
            contenido_limpio=re.sub(r"<think\b[^>]*>.*?</think\s*>","",contenido,flags=re.I|re.S).strip()
            residuo=bool(re.search(r"<\s*/?\s*think\b",contenido_limpio,re.I))
            registro["respuesta_final_evaluable"] = "" if residuo else contenido_limpio
            registro["razonamiento_separado"] = bool(msg.get("thinking"))
            registro["razonamiento_incrustado"] = original_tiene_think
            # DeepSeek usa think=True; su campo message.thinking es esperado,
            # pero no se muestra ni se usa para calificar la respuesta final.
            modo_pedido = bool(SOLICITUD_BASE.get("think", False))
            hay_pensamiento_separado = bool(msg.get("thinking"))
            if respuesta.get("done_reason") == "length":
                registro["estado_ejecucion"] = "truncada"
            elif residuo or original_tiene_think:
                registro["estado_ejecucion"] = "revisar_formato_modo"
            elif hay_pensamiento_separado != modo_pedido:
                registro["estado_ejecucion"] = "revisar_modo"
            elif respuesta.get("done_reason") == "stop" and contenido_limpio:
                registro["estado_ejecucion"] = "completa"
            else:
                registro["estado_ejecucion"] = "revisar"
        except Exception as e:
            registro["estado_ejecucion"] = "error"
            registro["error_tipo"] = type(e).__name__
            registro["error"] = str(e)
        registro["tiempo_cliente_s"] = round(time.perf_counter()-t0,3)
        registro["fecha_fin_utc"] = datetime.now(timezone.utc).isoformat()
        with SALIDA.open("a",encoding="utf-8") as f:
            f.write(json.dumps(registro,ensure_ascii=False)+"\n")
        respuesta=registro.get("respuesta_original",{})
        print(cid,"|",registro["estado_ejecucion"],"|",respuesta.get("eval_count","-"),"tokens |",registro["tiempo_cliente_s"],"s",flush=True)
        if registro["estado_ejecucion"] != "completa":
            print("Ejecución detenida: conserva el archivo y revisa este caso antes de continuar.")
            break
    print("Archivo actualizado:",SALIDA)

Etapa: restantes | programadas: 4 | ya completas: 0 | pendientes: 4
Ejecutando D2-008 | V4d
D2-008 | completa | 584 tokens | 685.341 s
Ejecutando D2-007 | V4d
D2-007 | completa | 553 tokens | 156.748 s
Ejecutando D2-006 | V4d
D2-006 | completa | 605 tokens | 783.298 s
Ejecutando D2-005 | V4d
D2-005 | error | - tokens | 900.011 s
Ejecución detenida: conserva el archivo y revisa este caso antes de continuar.
Archivo actualizado: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\05_Pruebas\02_Ronda2_V4d\Qwen3.8-27B\resultados_v4d.jsonl
